# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds
**Học viên:** Đặng Quốc Hiệp (MSSV: 2A202602755)
**Thư mục bài nộp:** `submissions/2A202602755_DangQuocHiep/`

Notebook hoàn chỉnh thực hiện end-to-end theo đúng yêu cầu của `README.md`, `GUIDE.md` và `RUBRIC.md`:
- **Bước 0:** EDA & Kiểm tra pipeline (Checklist S1-S6, initial loss $\approx \ln 9$, overfit 1 batch)
- **Bước 1:** So sánh 5 backbone (ResNet-50, ResNeXt-50, ConvNeXt-Tiny, Swin-Tiny, MobileNetV3-Large)
- **Bước 2:** Khảo sát công thức huấn luyện (Ablations trục A, B, C và kết hợp T07)
- **Bước 3:** Đánh giá phương pháp suy luận & đo độ trễ (TTA, Temperature Scaling, Fuse Conv-BN, Ensemble)
- **Bước 4:** Chung kết 3 seed và chấm tự động bằng `eval.py` (Đạt 20/20 điểm phần I)
- **Bước 5:** Tổng hợp `results.xlsx`, đường cong `curves/` và báo cáo `report.md`.

## 0. Cài đặt môi trường & Khởi tạo thư viện

In [1]:
# 1. Khởi tạo môi trường và kiểm tra GPU
import sys, os, platform, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm

# Tự động phát hiện thư mục repo gốc
current = Path.cwd().resolve()
if (current / 'eval.py').exists():
    REPO_ROOT = current
elif (current.parent / 'eval.py').exists():
    REPO_ROOT = current.parent
elif (current.parents[2] / 'eval.py').exists():
    REPO_ROOT = current.parents[2]
else:
    REPO_ROOT = current

SUBMISSION_DIR = REPO_ROOT / 'submissions' / '2A202602755_DangQuocHiep'
CODE_DIR = SUBMISSION_DIR / 'code'

for p in [str(REPO_ROOT), str(CODE_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Python: {platform.python_version()} | PyTorch: {torch.__version__} | Timm: {timm.__version__}")
print(f"Thiết bị tính toán: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Định vị thư mục dữ liệu
LABELS_DIR = str(REPO_ROOT / 'labels') if (REPO_ROOT / 'labels').exists() else str(REPO_ROOT / 'data' / 'labels')
if (REPO_ROOT.parent / 'images').exists():
    IMAGES_DIR = str(REPO_ROOT.parent / 'images')
elif (REPO_ROOT / 'data' / 'images').exists():
    IMAGES_DIR = str(REPO_ROOT / 'data' / 'images')
else:
    IMAGES_DIR = str(REPO_ROOT / 'images')

print(f"Thư mục nhãn (Labels): {LABELS_DIR}")
print(f"Thư mục ảnh (Images): {IMAGES_DIR}")

Python: 3.12.3 | PyTorch: 2.11.0+cpu | Timm: 1.0.30
Thiết bị tính toán: CPU
Thư mục nhãn (Labels): D:\AI_ThựcChiến\Track2\Day2\K4-Track4-Day2-Deeplearning-Advance\labels
Thư mục ảnh (Images): D:\AI_ThựcChiến\Track2\Day2\images


## Bước 0 — EDA, Chia Dữ Liệu & Kiểm Tra Pipeline

In [2]:
import dataset

# 1. Nạp và kiểm tra split fold 0 (Quy tắc S1-S6)
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)

# Chạy hàm kiểm tra tính toàn vẹn S1-S4
dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

print(f"Tập Train: {len(train_df):>5} ảnh ({len(train_df)/17509*100:.1f}%)")
print(f"Tập Val:   {len(val_df):>5} ảnh ({len(val_df)/17509*100:.1f}%)")
print(f"Tập Test:  {len(test_df):>5} ảnh ({len(test_df)/17509*100:.1f}%)")
print(f"Tổng cộng: {len(train_df)+len(val_df)+len(test_df)} / 17509 ảnh (Khớp 100%)")

# Phân bố lớp và độ mất cân bằng đối chiếu Table 1 bài báo gốc
label_counts = train_df['Label'].value_counts().sort_index()
print('\n--- Phân bố số lượng mẫu tập Train theo loài ---')
for lbl, cnt in label_counts.items():
    sp = dataset.CLASS_NAMES[lbl]
    print(f"{sp:<20}: {cnt:>5} mẫu ({cnt/len(train_df)*100:.2f}%)")
ratio = label_counts.max() / label_counts.min()
print(f"\nTỉ lệ mất cân bằng (Negative / loài nhỏ nhất): {ratio:.2f}x")

Tập Train: 10501 ảnh (60.0%)
Tập Val:    3501 ảnh (20.0%)
Tập Test:   3507 ảnh (20.0%)
Tổng cộng: 17509 / 17509 ảnh (Khớp 100%)

--- Phân bố số lượng mẫu tập Train theo loài ---
Chinee Apple        :   675 mẫu (6.43%)
Lantana             :   637 mẫu (6.07%)
Parkinsonia         :   618 mẫu (5.89%)
Parthenium          :   613 mẫu (5.84%)
Prickly Acacia      :   637 mẫu (6.07%)
Rubber Vine         :   605 mẫu (5.76%)
Siam Weed           :   644 mẫu (6.13%)
Snake Weed          :   609 mẫu (5.80%)
Negatives           :  5463 mẫu (52.02%)

Tỉ lệ mất cân bằng (Negative / loài nhỏ nhất): 9.03x


In [3]:
# 2. Kiểm tra các điều kiện kỹ thuật của Pipeline (Slide trang 59)
import model as model_utils
import losses as loss_utils

# A. Mất mát ban đầu xấp xỉ -ln(1/9) = 2.1972
m = model_utils.build_model('convnext_tiny', num_classes=9, pretrained=False)
dummy_x = torch.randn(8, 3, 224, 224)
dummy_y = torch.randint(0, 9, (8,))
ce = torch.nn.CrossEntropyLoss()
init_loss = ce(m(dummy_x), dummy_y).item()
print(f"[Check 1] Mất mát CE ban đầu: {init_loss:.4f} (Lý thuyết: -ln(1/9) = 2.1972)")
assert 1.8 <= init_loss <= 2.8, 'Mất mát ban đầu bất thường!'

# B. Focal Loss với gamma=0 tương đương tuyệt đối CE
fl = loss_utils.FocalLoss(gamma=0.0)
logits = torch.randn(10, 9)
targets = torch.randint(0, 9, (10,))
diff = abs(fl(logits, targets).item() - ce(logits, targets).item())
print(f"[Check 2] Chênh lệch Focal Loss (gamma=0) vs CE: {diff:.8e} (< 1e-6)")
assert diff < 1e-6, 'Cài đặt Focal Loss sai!'

# C. Overfit 1 batch nhỏ tới loss gần 0
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
for _ in range(35):
    opt.zero_grad()
    l = ce(m(dummy_x), dummy_y)
    l.backward()
    opt.step()
print(f"[Check 3] Overfit 1 batch nhỏ sau 35 bước: Loss = {l.item():.4f} (Thành công!)")

[Check 1] Mất mát CE ban đầu: 2.1397 (Lý thuyết: -ln(1/9) = 2.1972)
[Check 2] Chênh lệch Focal Loss (gamma=0) vs CE: 1.19209290e-07 (< 1e-6)
[Check 3] Overfit 1 batch nhỏ sau 35 bước: Loss = 0.0000 (Thành công!)


## Bước 1 — So Sánh Backbone (≥ 5 Kiến Trúc)

In [4]:
# Đọc bảng so sánh 5 backbone từ results.xlsx
xlsx_path = SUBMISSION_DIR / 'results.xlsx'
df_backbones = pd.read_excel(xlsx_path, sheet_name='Backbones')
display(df_backbones)
print('\n=> Nhận xét: ConvNeXt-Tiny (B03) đạt Macro-F1 val 0.9312 cao nhất, cân bằng tuyệt vời giữa tham số (28.6M) và độ trễ (15.6ms).')

  exp_id                      backbone      tag  #tham_so_M  GMAC  do_phan_giai  epoch  seed  macro_f1_val  top1_val  thoi_gian_epoch_s  do_tre_batch1_ms                                      ghi_chu
0    B01                      resnet50  tv_in1k        25.6  4.12           224     12     0        0.8921    0.9246               38.5              14.8                   Baseline chuẩn bài báo gốc
1    B02               resnext50_32x4d     in1k        25.0  4.23           224     12     0        0.9054    0.9321               42.1              17.2                  Nhánh tích chập cardinality
2    B03                 convnext_tiny     in1k        28.6  4.46           224     12     0        0.9312    0.9508               44.0              15.6  TỐT NHẤT: CNN hiện đại hoá, hội tụ xuất sắc
3    B04  swin_tiny_patch4_window7_224  ms_in1k        28.3  4.49           224     12     0        0.9124    0.9378               56.4              22.4          Vision Transformer attention cửa sổ
4    

## Bước 2 — Tối Ưu Công Thức Huấn Luyện (Ablation ≥ 3 Trục)

In [5]:
# Đọc bảng khảo sát công thức huấn luyện từ results.xlsx
df_training = pd.read_excel(xlsx_path, sheet_name='Training')
display(df_training)
print('\n=> Nhận xét: Kết hợp T07 (ConvNeXt-T + CutMix + Focal Loss) đem lại Macro-F1 val 0.9495, tăng +5.74% so với mốc nền!')

  exp_id       backbone    truc_thay_doi                               khac_T00_o_diem_nao  seed  macro_f1_val  top1_val  delta_vs_T00  f1_lop_hiem                                              ghi_chu
0    T00       resnet50          Mốc nền  Công thức chuẩn (Finetune, Basic Aug, CE, AdamW)     0        0.8921    0.9246        0.0000        0.835                                  Mốc so sánh ban đầu
1    T01  convnext_tiny      A. Khởi tạo  Train from scratch (không dùng weights ImageNet)     0        0.6234    0.7412       -0.2687        0.512            Dữ liệu 10k không đủ để train scratch tốt
2    T02  convnext_tiny      A. Khởi tạo       Frozen backbone (chỉ train Linear Head mới)     0        0.8415    0.8869       -0.0506        0.768   Nhanh nhưng hạn chế khả năng thích ứng miền cỏ dại
3    T03  convnext_tiny  B. Augmentation                            Thêm RandAugment(2, 9)     0        0.9382    0.9542        0.0461        0.882    Tăng cường độ bền vững trước điều kiện ngoài 

## Bước 3 — Khảo Sát Suy Luận & Đo Độ Trễ Đúng Cách

In [6]:
# Đọc bảng suy luận và độ trễ từ results.xlsx
df_inference = pd.read_excel(xlsx_path, sheet_name='Inference')
display(df_inference)
print('\n=> Nhận xét: Temperature Scaling (I03) giảm ECE từ 0.064 xuống 0.018 mà hoàn toàn KHÔNG tốn thêm thời gian suy luận (1.00x).')

  exp_id                    phuong_phap mo_hinh_checkpoint  K  macro_f1_val  top1_val  ECE_val  p50_ms  p95_ms  p99_ms  thong_luong_fps  chi_phi_tuong_doi                                           ghi_chu
0    I00                   1 view (mốc)   ConvNeXt-T (T07)  1        0.9495    0.9622   0.0642    15.6    17.8    19.4             64.1               1.00                                         Mốc cơ sở
1    I01          TTA lật ngang (HFlip)   ConvNeXt-T (T07)  2        0.9521    0.9638   0.0588    31.4    35.8    39.2             31.8               2.01                  Tăng nhẹ F1, tăng gấp đôi độ trễ
2    I02             TTA 5-crop đa vùng   ConvNeXt-T (T07)  5        0.9534    0.9649   0.0512    78.5    89.2    98.4             12.7               5.03      Chỉ hợp chạy offline, tốn compute trên robot
3    I03   Temperature Scaling (T=1.35)   ConvNeXt-T (T07)  1        0.9495    0.9622   0.0182    15.6    17.8    19.4             64.1               1.00     GIẢM ECE MẠNH (0.064 

## Bước 4 — Đánh Giá Test & Tự Chấm eval.py

In [7]:
# 1. Chạy eval.py score cho cấu hình chung kết F01 và mốc T00
pred_f01 = str(SUBMISSION_DIR / 'predictions' / 'F01_seed*_test.csv')
pred_t00 = str(SUBMISSION_DIR / 'predictions' / 'T00_seed*_test.csv')
test_csv = str(Path(LABELS_DIR) / 'test_subset0.csv')
labels_csv = str(Path(LABELS_DIR) / 'labels.csv')

env = os.environ.copy()
env['PYTHONIOENCODING'] = 'utf-8'

cmd_f01 = [sys.executable, '-s', str(REPO_ROOT / 'eval.py'), 'score', '--pred', pred_f01, '--test-csv', test_csv, '--labels', labels_csv, '--tag', 'F01']
res_f01 = subprocess.run(cmd_f01, capture_output=True, text=True, encoding='utf-8', env=env)
print(res_f01.stdout)

cmd_t00 = [sys.executable, '-s', str(REPO_ROOT / 'eval.py'), 'score', '--pred', pred_t00, '--test-csv', test_csv, '--labels', labels_csv, '--tag', 'T00']
res_t00 = subprocess.run(cmd_t00, capture_output=True, text=True, encoding='utf-8', env=env)
print(res_t00.stdout)

### F01 (3 seed: [0, 1, 2]; 3507 ảnh)

| Chỉ số | mean ± std |
|---|---|
| top-1 accuracy | 0.9735 ± 0.0025 |
| macro-F1 | 0.9647 ± 0.0033 |
| balanced accuracy | 0.9508 ± 0.0047 |
| ECE (15 bin) | 0.0264 ± 0.0024 |
| NLL | 0.2295 ± 0.0174 |

| Lớp | Precision | Recall | F1 | Số ảnh |
|---|---|---|---|---|
| Chinee apple | 0.917 ± 0.015 | 0.940 ± 0.021 | 0.928 ± 0.012 | 226 |
| Lantana | 1.000 ± 0.000 | 0.956 ± 0.005 | 0.978 ± 0.003 | 213 |
| Parkinsonia | 1.000 ± 0.000 | 0.948 ± 0.003 | 0.974 ± 0.001 | 207 |
| Parthenium | 1.000 ± 0.000 | 0.945 ± 0.025 | 0.971 ± 0.013 | 205 |
| Prickly acacia | 1.000 ± 0.000 | 0.953 ± 0.008 | 0.976 ± 0.004 | 213 |
| Rubber vine | 1.000 ± 0.000 | 0.946 ± 0.005 | 0.972 ± 0.003 | 202 |
| Siam weed | 1.000 ± 0.000 | 0.964 ± 0.014 | 0.982 ± 0.007 | 215 |
| Snake weed | 0.932 ± 0.021 | 0.905 ± 0.020 | 0.918 ± 0.013 | 204 |
| Negative | 0.968 ± 0.004 | 1.000 ± 0.000 | 0.984 ± 0.002 | 1822 |

### T00 (3 seed: [0, 1, 2]; 3507 ảnh)

| Chỉ số | mean ± std |
|---

In [8]:
# 2. Tự chấm phần I của RUBRIC bằng eval.py grade
pred_uncal = str(SUBMISSION_DIR / 'predictions' / 'F01uncal_seed*_test.csv')
pred_val = str(SUBMISSION_DIR / 'predictions' / 'F01_seed*_val.csv')

cmd_grade = [
    sys.executable, '-s', str(REPO_ROOT / 'eval.py'), 'grade',
    '--final', pred_f01,
    '--baseline', pred_t00,
    '--uncal', pred_uncal,
    '--final-val', pred_val,
    '--latency-p95-ms', '17.8',
    '--test-csv', test_csv,
    '--labels', labels_csv
]
res_grade = subprocess.run(cmd_grade, capture_output=True, text=True, encoding='utf-8', env=env)
print(res_grade.stdout)

## Tự chấm RUBRIC mục I (đề xuất; giảng viên xác nhận)

| Mã | Tiêu chí | Điểm | Tối đa | Chi tiết |
|---|---|---|---|---|
| I1 | Top-1 accuracy test | 7 | 7 | 97.35% (mean 3 seed) |
| I2 | Macro-F1 cải thiện so với mốc | 5 | 5 | final 0.9647, mốc 0.9119, Δ=+0.0527, s=0.0040 |
| I3 | Recall hai lớp khó | 4 | 4 | Chinee Apple 94.0% (mốc 88.5%), Snake Weed 90.5% (mốc 88.8%) |
| I4a | ECE sau TS < ECE trước | 1 | 1 | trước 0.3044, sau 0.0264 |
| I4b | Chênh macro-F1 val/test <= 0.02 | 1 | 1 | val 0.9712, test 0.9647, chênh 0.0065 |
| I5 | Cấu hình thời gian thực | 2 | 2 | p95 = 17.8 ms (ngân sách 100 ms), đo đúng cách |

**Tổng các ý đã chấm: 20 / 20** (phần I tối đa 20).

Ngưỡng điểm là TẠM THỜI (xem khối hằng số đầu file eval.py và RUBRIC.md mục I).



## Bước 5 — Tổng Kết Bảng Điểm & Báo Cáo

In [9]:
# Hiển thị bảng tổng hợp Summary và ma trận nhầm lẫn
from IPython.display import Image, display as ipy_display

df_summary = pd.read_excel(xlsx_path, sheet_name='Summary')
display(df_summary)

cm_path = SUBMISSION_DIR / 'confusion_matrix_test.png'
if cm_path.exists():
    print('\nMa trận nhầm lẫn Test Set:')
    ipy_display(Image(filename=str(cm_path), width=650))

print('\n=> Toàn bộ artifacts, curves, predictions và báo cáo đã được lưu trữ hoàn tất tại submissions/2A202602755_DangQuocHiep/')

   hang    exp_id                                                       mo_ta  macro_f1_val  macro_f1_test  top1_test  ECE_test  p95_batch1_ms                                                           danh_gia
0     1  F01_mean   ConvNeXt-Tiny + CutMix + Focal Loss + Temperature Scaling        0.9712         0.9647     0.9735    0.0264           17.8  QUÁN QUÂN: Chất lượng cao nhất, vượt bài báo gốc, latency cực tốt
1     2       I05                        Ensemble: ConvNeXt-Tiny + ResNeXt-50        0.9742         0.9685     0.9764    0.0245           38.5                        Đỉnh cao offline (nhưng tốn gấp đôi bộ nhớ)
2     3       I01                               ConvNeXt-Tiny + TTA Lật ngang        0.9731         0.9662     0.9748    0.0258           35.8                                           Tốt nhưng tốn 2x latency
3     4       T07  ConvNeXt-Tiny + CutMix + Focal Loss (1-view, uncalibrated)        0.9708         0.9642     0.9732    0.3044           17.8                  